# NIH ChestX-ray14: resize to 224px on Colab

**For education only, not for clinical use.**

Downloads the NIH dataset from Kaggle (`nih-chest-xrays/data`) to the Colab VM, resizes every image to
224x224 8-bit grayscale PNG and writes them to Drive in zip shards of 2,000 images, with a checksum
manifest.

- **Not run yet.** The code it calls (`src/data/preprocess.py`) is unit-tested on synthetic images only.
- Use a **CPU runtime**: no GPU is needed, and CPU runtimes have more disk. The dataset zip is about
  45 GB on the VM. It is read directly and never extracted.
- **Raw data never goes to Drive.** Drive gets only the resized shards and `shards.csv` / `images.csv`.
- **Resumable:** after a disconnect, run all cells again. The download starts over (the VM disk is
  wiped), but shards already in Drive are checked against their sha256 and skipped.
- Kaggle credentials are optional: this public dataset downloaded without them from a local
  machine (2026-11-03). If Kaggle asks for them, add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab
  secrets.

## 1. Drive and paths

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

OUT = Path("/content/drive/MyDrive/medical-imaging-ai/nih_224")
OUT.mkdir(parents=True, exist_ok=True)

# Raw data and temporary files stay on the VM.
RAW = Path("/content/raw")
WORK = Path("/content/work")
for path in (RAW, WORK):
    assert not str(path).startswith("/content/drive"), f"{path} must not be on Drive"
    path.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage("/content").free / 1e9
print(f"VM disk free: {free_gb:.1f} GB")
assert any(RAW.glob("*.zip")) or free_gb > 55, "Need about 55 GB free; use a CPU runtime"

## 2. Code from the repo

In [ ]:
REPO_URL = "https://github.com/manojkumargudge/medical-imaging-ai.git"
REPO_REF = "main"
REPO = Path("/content/medical-imaging-ai")

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True
    )
# On the path rather than installed: the repo pins are for local Python 3.11, and Colab's own
# Pillow is enough here.
sys.path.insert(0, str(REPO))
commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit", commit)

from src.data import preprocess

## 3. Kaggle CLI (credentials optional)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.5", "six==1.17.0"], check=True)

try:
    from google.colab import userdata

    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Using Kaggle credentials from Colab secrets")
except Exception as exc:
    print(f"No Kaggle secrets ({type(exc).__name__}); trying without credentials")

## 4. Verify the dataset slug and file names

In [ ]:
SLUG = "nih-chest-xrays/data"

listing = subprocess.run(
    ["kaggle", "datasets", "files", SLUG, "--page-size", "20"],
    capture_output=True,
    text=True,
    check=True,
)
print(listing.stdout)
assert "Data_Entry_2017.csv" in listing.stdout, f"Unexpected listing for {SLUG}: {listing.stderr}"

## 5. Download to the VM (about 45 GB, not extracted)

In [ ]:
if not any(RAW.glob("*.zip")):
    subprocess.run(["kaggle", "datasets", "download", SLUG, "-p", str(RAW)], check=True)

zips = sorted(RAW.glob("*.zip"))
assert len(zips) == 1, f"Expected one zip, found {zips}"
print(zips[0], f"{zips[0].stat().st_size / 1e9:.1f} GB")

## 6. Check the image count

In [ ]:
EXPECTED_IMAGES = 112_120

images = preprocess.ZipImages(zips[0])
names = images.names
print(f"PNG images in zip: {len(names)}")
assert len(names) == EXPECTED_IMAGES, f"Expected {EXPECTED_IMAGES} images, found {len(names)}"

## 7. Resize and write shards to Drive

About 56 shards. Finished shards are skipped on a rerun after their sha256 is checked. A shard that
exists in Drive but differs from what this run would write is never overwritten: the run stops.

In [ ]:
report = preprocess.write_shards(names, images.read, OUT, WORK)
images.close()
print(report)

## 8. Verify everything in Drive and report the size

In [ ]:
import json

verified = preprocess.verify_shards(OUT)
print(f"zip images {len(names)}, images in Drive shards {verified.images}")
assert len(names) > 0 and verified.images > 0
assert verified.images == len(names), "Some images are missing from the shards"
print(f"{verified.shards} shards, {verified.total_bytes / 1e9:.2f} GB on Drive")

(OUT / "run.json").write_text(
    json.dumps(
        {
            "repo_commit": commit,
            "images": verified.images,
            "shards": verified.shards,
            "total_bytes": verified.total_bytes,
            "image_size": preprocess.IMAGE_SIZE,
        },
        indent=2,
    )
)

In [ ]:
# Spot check: first image of the first shard.
import io
import zipfile

import matplotlib.pyplot as plt
from PIL import Image

with zipfile.ZipFile(OUT / "shard-00000.zip") as archive:
    first = archive.namelist()[0]
    image = Image.open(io.BytesIO(archive.read(first)))
    print(first, image.mode, image.size)
    plt.imshow(image, cmap="gray")
    plt.title(first)
    plt.axis("off")
    plt.show()